# **Proyecto 05: Sistema de Identificación del Género de una Canción**

---

## **Parte 1: Carga de los datos**

En este proyecto, vamos a construir una red neuronal artificial que puede identificar el género de una canción, utilizando la librería GTZAN Genre Collection. Para extraer los features (características) de las canciones, vamos a utilizar la librería de Python [**librosa**](https://librosa.org/). Utilizaremos los Mel-frequency cepstral coefficients (MFCC), que simulan la escucha humana y son coúnmente utilizados en aplicaciones de speech recognition (reconocimiento del habla) así como en detección del género musical. Estos valores serán los que utilizaremos como entrada de la red neuronal.

Para entender qué son los MFCC, podemos descargar Kick Loop 5 by Stereo Surgeon desde [https://freesound.org/people/Stereo%20Surgeon/sounds/266093](https://freesound.org/people/Stereo%20Surgeon/sounds/266093), y también descargar Whistling by cmagar desde [https://freesound.org/people/grrlrighter/sounds/98195/](https://freesound.org/people/grrlrighter/sounds/98195/). Uno de ellos es un beat de bajas frecuencias, mientras que el otros es un silbido de frecuencias más agudas. Veremos mediante los valores MFCC como estos dos sonidos son claramente distintos.

Además de importar la librería librosa, también vamos a usar [**glob**](https://docs.python.org/es/3/library/glob.html) para poder hacer un listado de los archivos en los diferentes repositorios de los géneros musicales. Además, utilizaremos <span style="color:blue"> **numpy**</span> y <span style="color:blue"> **matplotlib**</span>.

También vamos a importar el modelo **Sequential** de **Keras**. Este es un modelo típico de red neuronal feed-forward. Finalmente, importaremos una capa densa de una red neuronal (capa con una colección de neuronas en ella, [**Dense**](https://keras.io/api/layers/core_layers/dense/https://keras.io/api/layers/core_layers/dense/)).


A diferencia de con las operaciones convolución, por ejemplo, esta red va a tener representaciones en dos dimensiones. Vamos a importar funciones de activación, lo que nos permitirá decidir en cada capa de la red neuronal que función no lineal utilizar, y también importaremos la función <span style="color:blue"> **to_categorical**</span>, que nos permite cambiar los nombres de las clases en categorías, que es lo que ocurre con el one-hot encoding. 

En primer lugar, vamos a definir una función llamada <span style="color:blue"> **display_mfcc**</span> que nos permitirá visualizar en un gráfico estos valores. Los MFCC se pueden obtener a partir de la librería librosa. Para visualizar, utilizaremos un tipo de gráfico conocido como espectrograma, que pertenece a la libraría librosa (<span style="color:blue"> **specshow**</span>).

Vamos a probar esta función que hemos creado con las canciones 'kick-loop.wav' y 'whistling.wav' que hemos importado previamente. 

---
## **Parte 2: Preprocesado de los datos**

Ahora que ya podemos visualizar los coeficientes MCFF, vamos a crear otra función auxiliar, que nos permitirá obtener dichos coeficientes y guardarlos en un vector. Para mejor funcionamiento de la posterior red neuronal, vamos a normalizar los coeficients para que su rango esté comprendido entre -1 y 1, y solamente vamos a utilizar los primeros 25000 coeficientes. A la función auxiliar la vamos a llamar  <span style="color:blue"> **extract_features_song**</span> 

A continuación vamos a definir una rutina para abrir los ficheros con las canciones de los distintos géneros y extraer los coeficientes MCFF de ellas. La función se llamará  <span style="color:blue"> **generate_features_and_labels**</span>, y en ella haremos un bucle sobre todos los géneros, y buscaremos en la carpeta de cada género todos los archivos, los abriremos y extraeremos los MCFF mediante la función extract_features_song. Al mismo tiempo, crearemos un vector de etiquetas (labels) y añadiremos el género a ese vector cada vez que abramos un fichero. Sin embargo, la red neuronal no es capaz de predecir una palabra o letras. Para poder utilizarla, necesitamos hacer un one-hot encoding, lo que significa que cada palabra será representada por un vector de diez números binarios (tenemos 10 clases en total), de manera que solamente uno de los números sea 1 y el resto sean 0. Utilizaremos la función  <span style="color:blue"> **np.unique**</span> para hacer que las etiquetas se conviertan en números enteros. Luego, utilizamos la función  <span style="color:blue"> **to_categorical**</span>, que convertirá esos números enteros en representación one-hot encoding. Por último, utilizamos la función  <span style="color:blue"> **np.stack**</span> sobre los features que hemos extraído de las canciones para que se junten en una única matriz.    

Vamos a dividir el dataset de features y labels en train y test, dejando un 80% de los datos para el train y un 20% para el test. Antes de dividir, hay que primero juntar los datos de los features y las etiquetas utilizando la función  <span style="color:blue"> **np.column_stack**</span>, luego  hay que barajar los datos para no producir artefactos debido al orden de los datos en el modelo, para ello, podemos utilizar la función  <span style="color:blue"> **shuffle**</span>. Una vez separados en train y test, debemos eliminar de cada set las 10 últimas columnas, puesto que son las columnas correspondientes al one-hot encoding, que no necesitamos como entrada, si no como salida de nuestra red neuronal.

---
## **Parte 3: Análisis del modelo (entrenamiento de la red neuronal)**

Seguidamente construimos la red neuronal. Vamos a utilizar un modelo secuencial ( <span style="color:blue"> **Sequential**</span>). Añadiremos una capa densa de 100 neuronas en primer lugar, definiendo también en esta primera capa el tamaño de entrada de la red (que podemos obtener a partir del tamaño del train dataset). La función de activación que escogeremos para la primera capa es la función  <span style="color:red"> 'relu'</span>, y luego añadimos una segunda capa densa de 10 neuronas con una función de activación de tipo  <span style="color:red"> 'softmax'</span>. El tamaño de la segunda capa viene fijado por el número de clases que tenemos en nuestro clasificador, es decir 10 géneros musicales distintos. La activación  <span style="color:red"> 'softmax'</span> coge las 10 salidas y las normaliza de tal forma que la suma de las 10 sea 1. De esta forma, acaban representando probabilidades, así que la predicción será aquella que tenga una probabilidad mayor. 

Una vez creado el modelo, debemos compilarlo (instrucción [**compile**](https://www.tensorflow.org/api_docs/python/tf/keras/Model)), añadiendo algunos hiperparámetros de como funciona nuestro algoritmo. En concreto, vamos a utilizar el optimizador  <span style="color:red"> 'adam'</span>, y la función de pérdidas será  <span style="color:red"> 'categorical_crossentropy'</span>. Por último, la medida que queremos ver durante la evaluación es la medida de  <span style="color:red"> 'accuracy'</span>. Par apoder ver un resumen de la arquitectura creada, podemos mostrar por pantalla un resumen <span style="color:blue"> **model.summary**</span> que nos da los detalles sobre las capas.

Por último, solamente tenemos que entrenar el model [**fit**](https://www.tensorflow.org/api_docs/python/tf/keras/Model) definiendo como entradas los datos de entrenamiento (tanto los features como las etiquetas, y definimos el número de épocas (10 por ejemplo), el tamaño del batch (32 por ejemplo) y el tamaño del conjunto de validación (0.2 por ejemplo). Por último, obtendremos las métricas de nuestro sistema evaluando la precisión mediante los datos de test con la función [**evaluate**](https://www.tensorflow.org/api_docs/python/tf/keras/Model).